In [18]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

In [19]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
89,"Although I'm a girl, thankfully I have a sense...",positive
187,OK - you want to test somebody on how comforta...,positive
656,i watch this film with horror in my heart beca...,positive
273,No movies have grabbed my attention like this ...,positive
516,"Back in 1994, I had a really lengthy vacation ...",positive


In [20]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [21]:
df = normalize_text(df)
df.head()

,review,sentiment
89,although girl thankfully sense humor realize r...,positive
187,ok want test somebody comfortable adolescence ...,positive
656,watch film horror heart mother also crack head...,positive
273,movie grabbed attention like one ha see wanted...,positive
516,back really lengthy vacation around fourth jul...,positive


In [22]:
df['sentiment'].value_counts()

sentiment
negative    260
positive    240
Name: count, dtype: int64

In [23]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [24]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
89,although girl thankfully sense humor realize r...,1
187,ok want test somebody comfortable adolescence ...,1
656,watch film horror heart mother also crack head...,1
273,movie grabbed attention like one ha see wanted...,1
516,back really lengthy vacation around fourth jul...,1


In [25]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [ ]:
vectorizer = CountVectorizer(max_features=1000)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.75, random_state=75)

In [28]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/SANJAY-SRINIVAS226/Capstone-project.mlflow')
dagshub.init(repo_owner='SANJAY-SRINIVAS226', repo_name='Capstone-project', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")

Initialized MLflow to track repo "SANJAY-SRINIVAS226/Capstone-project"

2026-06-04 18:09:41,279 - INFO - Initialized MLflow to track repo "SANJAY-SRINIVAS226/Capstone-project"


Repository SANJAY-SRINIVAS226/Capstone-project initialized!

2026-06-04 18:09:41,283 - INFO - Repository SANJAY-SRINIVAS226/Capstone-project initialized!


<Experiment: artifact_location='mlflow-artifacts:/bb348505c2eb41e7be1a51850d1a38a7', creation_time=1780576618868, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1780576618868, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, trace_location=None, workspace='default'>

In [ ]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 1000)
        mlflow.log_param("test_size", 0.75)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)

2026-06-04 18:09:42,328 - INFO - Starting MLflow run...
2026-06-04 18:09:42,885 - INFO - Logging preprocessing parameters...
2026-06-04 18:09:44,078 - INFO - Initializing Logistic Regression model...
2026-06-04 18:09:44,080 - INFO - Fitting the model...
2026-06-04 18:09:44,118 - INFO - Model training complete.
2026-06-04 18:09:44,118 - INFO - Logging model parameters...
2026-06-04 18:09:44,462 - INFO - Making predictions...
2026-06-04 18:09:44,464 - INFO - Calculating evaluation metrics...
2026-06-04 18:09:44,480 - INFO - Logging evaluation metrics...
2026-06-04 18:09:46,035 - INFO - Saving and logging the model...
2026/06/04 18:09:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/04 18:09:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The rec

🏃 View run unequaled-conch-568 at: https://dagshub.com/SANJAY-SRINIVAS226/Capstone-project.mlflow/#/experiments/0/runs/705e601cba1d4e7d90f105dfa5446315
🧪 View experiment at: https://dagshub.com/SANJAY-SRINIVAS226/Capstone-project.mlflow/#/experiments/0
